# SSO Signup Optimization V2 — Data Analysis

This notebook builds up, CTE by CTE, the query used to establish the warehouse baseline for this experiment's power analysis: the **First Fix conversion baseline**. Each step below adds exactly one CTE and re-runs it, so the final step reproduces the exact query and numbers used for sizing.

The query anchors on a visitor's first visit to the signup page each month, on desktop or mobile web, and then asks whether that visitor went on to request a First Fix within 7 days.

In [1]:
import pandas as pd
from amphibian import get_data_accessor

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

## Building the First Fix conversion baseline

Visits from `2026-01-01` onward, to any URL starting with `https://www.stitchfix.com/signup`, restricted to `platform = 'web'`. The signup page is a web-only surface (the iOS app uses a native signup flow), so this filter makes that scope explicit rather than relying on the URL pattern alone.

### Step 1 — the `signup_page_visits` CTE alone

One row per visitor per month, anchored to the **first** time they reached the signup page that month (not every visit — just the earliest one, which is what "new" is measured relative to).

In [2]:
query("""--sql
SELECT visitor_id, DATE_TRUNC('month', datetime_in_utc) AS month, MIN(datetime_in_utc) AS signup_page_ts
FROM curated.product_tracking_events
WHERE date_in_utc >= DATE '2026-01-01'
  AND url LIKE 'https://www.stitchfix.com/signup%'
  AND platform = 'web'
GROUP BY visitor_id, DATE_TRUNC('month', datetime_in_utc)
ORDER BY signup_page_ts
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,visitor_id,month,signup_page_ts
0,4a4eda80-25ef-4e13-bc8c-1f9a1ccc66e7,2026-01-01 00:00:00.000,2026-01-01 00:00:03.030
1,15a3e35b-3505-4448-bd24-b11f0f715372,2026-01-01 00:00:00.000,2026-01-01 00:00:03.750
2,c7fff5a0-6feb-491e-824d-09bdd702c86e,2026-01-01 00:00:00.000,2026-01-01 00:00:22.407
3,0f1f2a01-4bde-40bf-a211-e3b4a040d8ff,2026-01-01 00:00:00.000,2026-01-01 00:00:25.349
4,9b4b1515-af03-4221-9410-b2f7dcf762c6,2026-01-01 00:00:00.000,2026-01-01 00:00:32.175


In [3]:
query("""--sql
SELECT DATE_TRUNC('month', datetime_in_utc) AS month, COUNT(DISTINCT visitor_id) AS distinct_visitors
FROM curated.product_tracking_events
WHERE date_in_utc >= DATE '2026-01-01'
  AND url LIKE 'https://www.stitchfix.com/signup%'
  AND platform = 'web'
GROUP BY 1
ORDER BY 1 DESC
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,distinct_visitors
0,2026-08-01 00:00:00.000,260945
1,2026-07-01 00:00:00.000,345317
2,2026-06-01 00:00:00.000,259772
3,2026-05-01 00:00:00.000,311574
4,2026-04-01 00:00:00.000,334116
5,2026-03-01 00:00:00.000,423476
6,2026-02-01 00:00:00.000,386568
7,2026-01-01 00:00:00.000,412145


These monthly counts are **every** visitor who reached the signup page that month — before any eligibility gate is applied. July 2026's count is larger than what the final query below reports for July, because this hasn't yet excluded visitors who had already converted before this particular visit — that gate is added next.

### Step 2 — the `visitor_conversion` CTE alone

One row per visitor, pulling both `signup_ts` and `request_7d_flag` from `curated.user_session_conversion_metrics` — a visitor-level fact independent of any specific page visit. `request_7d_flag` is the table's own precomputed "requested a First Fix within 7 days" flag; a visitor can have multiple sessions in this table, so it's aggregated with `MAX()` (1 if *any* session carries the flag). This is what lets the next step tell whether a signup-page visit was a genuinely new reach, and whether that visitor went on to request a First Fix.

In [4]:
query("""--sql
SELECT visitor_id, MAX(signup_ts) AS signup_ts, MAX(COALESCE(request_7d_flag, 0)) AS first_fix_request_7d_flag
FROM curated.user_session_conversion_metrics
WHERE region = 'US'
  AND date_in_utc >= DATE '2026-01-01'
GROUP BY visitor_id
HAVING MAX(COALESCE(request_7d_flag, 0)) = 1
ORDER BY signup_ts
LIMIT 5
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,visitor_id,signup_ts,first_fix_request_7d_flag
0,e8e82847-1fe2-4b7e-b937-a3927e75321d,2011-10-30 07:00:00.000,1
1,9db97391-d09b-44af-a3fb-1ab9ced6dda1,2011-10-30 07:00:00.000,1
2,71f954ee-9754-4042-85a4-fe5e98a78462,2011-10-30 07:00:00.000,1
3,db0130b3-5f60-4282-b1f6-6fb825652815,2011-10-30 07:00:00.000,1
4,169d0c9c-87a8-4049-a084-93bcc3627266,2011-10-30 07:00:00.000,1


In [5]:
query("""--sql
SELECT
  COUNT(*) AS n_visitors,
  SUM(CASE WHEN signup_ts IS NOT NULL THEN 1 ELSE 0 END) AS n_with_signup,
  SUM(first_fix_request_7d_flag) AS n_with_first_fix_request
FROM (
  SELECT visitor_id, MAX(signup_ts) AS signup_ts, MAX(COALESCE(request_7d_flag, 0)) AS first_fix_request_7d_flag
  FROM curated.user_session_conversion_metrics
  WHERE region = 'US'
    AND date_in_utc >= DATE '2026-01-01'
  GROUP BY visitor_id
)
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_visitors,n_with_signup,n_with_first_fix_request
0,74872704,6841335,1560781


A small fraction of all visitors in this window ever carry the First Fix flag — a much smaller slice than the number who ever signed up, which is expected: requesting a Fix is a deeper funnel step than signing up, so it's a strict subset. There's no raw "request timestamp" column on this table (only the precomputed 7-day flag), so this flag is taken at face value rather than re-derived against the exact visit timestamp.

### Step 3 — the `classified` CTE: joining the two together

Left join `signup_page_visits` to `visitor_conversion` and derive two flags per visit:
- **`new_visitor`**: 1 if the visitor hadn't signed up yet as of this visit (i.e. this is a genuine "new" reach, not a returning member re-visiting the page).
- **`first_fix_request`**: 1 if the visitor's `request_7d_flag` is set.

Five real visitors, run through this logic, illustrate the cases:

In [6]:
query(f"""--sql
WITH signup_page_visits AS (
    SELECT visitor_id, DATE_TRUNC('month', datetime_in_utc) AS month, MIN(datetime_in_utc) AS signup_page_ts
    FROM curated.product_tracking_events
    WHERE date_in_utc >= DATE '2026-01-01'
      AND url LIKE 'https://www.stitchfix.com/signup%'
      AND platform = 'web'
      AND visitor_id IN ('4a4eda80-25ef-4e13-bc8c-1f9a1ccc66e7', '15a3e35b-3505-4448-bd24-b11f0f715372', 'c7fff5a0-6feb-491e-824d-09bdd702c86e', '0f1f2a01-4bde-40bf-a211-e3b4a040d8ff', '9b4b1515-af03-4221-9410-b2f7dcf762c6')
    GROUP BY visitor_id, DATE_TRUNC('month', datetime_in_utc)
),
visitor_conversion AS (
    SELECT visitor_id, MAX(signup_ts) AS signup_ts, MAX(COALESCE(request_7d_flag, 0)) AS first_fix_request_7d_flag
    FROM curated.user_session_conversion_metrics
    WHERE region = 'US' AND date_in_utc >= DATE '2026-01-01'
      AND visitor_id IN ('4a4eda80-25ef-4e13-bc8c-1f9a1ccc66e7', '15a3e35b-3505-4448-bd24-b11f0f715372', 'c7fff5a0-6feb-491e-824d-09bdd702c86e', '0f1f2a01-4bde-40bf-a211-e3b4a040d8ff', '9b4b1515-af03-4221-9410-b2f7dcf762c6')
    GROUP BY visitor_id
)
SELECT
    v.visitor_id, v.signup_page_ts, c.signup_ts,
    CASE WHEN c.signup_ts IS NULL OR c.signup_ts >= v.signup_page_ts THEN 1 ELSE 0 END AS new_visitor,
    COALESCE(c.first_fix_request_7d_flag, 0) AS first_fix_request
FROM signup_page_visits v
LEFT JOIN visitor_conversion c ON c.visitor_id = v.visitor_id
ORDER BY v.signup_page_ts
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,visitor_id,signup_page_ts,signup_ts,new_visitor,first_fix_request
0,4a4eda80-25ef-4e13-bc8c-1f9a1ccc66e7,2026-01-01 00:00:03.030,2022-05-13 22:52:03.740,0,0
1,15a3e35b-3505-4448-bd24-b11f0f715372,2026-01-01 00:00:03.750,None,1,0
2,c7fff5a0-6feb-491e-824d-09bdd702c86e,2026-01-01 00:00:22.407,None,1,0
3,0f1f2a01-4bde-40bf-a211-e3b4a040d8ff,2026-01-01 00:00:25.349,2026-01-01 00:02:00.184,1,0
4,9b4b1515-af03-4221-9410-b2f7dcf762c6,2026-01-01 00:00:32.175,2026-01-01 00:00:46.555,1,0


Reading each row: a visitor with a `signup_ts` from years before this visit gets `new_visitor = 0` (a returning member, not a fresh reach; excluded from the funnel entirely). Visitors with `signup_ts = NULL` have `new_visitor = 1, first_fix_request = 0` if they never signed up. None of these five particular visitors requested a Fix within the window, reflecting how rare the event is at the individual level: two of the five never signed up at all, and the others signed up only minutes before this snapshot, leaving little time to also complete Style Profile and request a Fix inside a 7-day window. This is exactly the population/eligibility logic the final query aggregates over.

### Step 4 — the full First Fix conversion query

Add `month_days` (distinct calendar days observed per month, for the daily-rate denominator) and the final aggregation. This is the complete query used for the First Fix conversion baseline.

In [7]:
first_fix_query = """--sql
WITH signup_page_visits AS (
    SELECT
        visitor_id,
        DATE_TRUNC('month', datetime_in_utc) AS month,
        MIN(datetime_in_utc) AS signup_page_ts
    FROM curated.product_tracking_events
    WHERE date_in_utc >= DATE '2026-01-01'
      AND url LIKE 'https://www.stitchfix.com/signup%'
      AND platform = 'web'
    GROUP BY visitor_id, DATE_TRUNC('month', datetime_in_utc)
),
visitor_conversion AS (
    SELECT
        visitor_id,
        MAX(signup_ts) AS signup_ts,
        MAX(COALESCE(request_7d_flag, 0)) AS first_fix_request_7d_flag
    FROM curated.user_session_conversion_metrics
    WHERE region = 'US'
      AND date_in_utc >= DATE '2026-01-01'
    GROUP BY visitor_id
),
classified AS (
    SELECT
        v.month,
        v.signup_page_ts,
        CASE WHEN c.signup_ts IS NULL OR c.signup_ts >= v.signup_page_ts
             THEN 1 ELSE 0 END AS new_visitor,
        COALESCE(c.first_fix_request_7d_flag, 0) AS first_fix_request
    FROM signup_page_visits v
    LEFT JOIN visitor_conversion c ON c.visitor_id = v.visitor_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT CAST(signup_page_ts AS DATE)) AS days_observed
    FROM signup_page_visits
    GROUP BY month
)
SELECT
    c.month,
    md.days_observed,
    SUM(new_visitor) AS signup_page_visitors,
    SUM(CASE WHEN new_visitor = 1 THEN first_fix_request ELSE 0 END) AS first_fix_requests,
    CAST(SUM(CASE WHEN new_visitor = 1 THEN first_fix_request ELSE 0 END) AS DOUBLE)
        / NULLIF(SUM(new_visitor), 0) AS first_fix_conv_rate,
    ROUND(SUM(new_visitor) / md.days_observed, 1) AS signup_page_visitors_per_day
FROM classified c
JOIN month_days md ON c.month = md.month
GROUP BY c.month, md.days_observed
ORDER BY c.month DESC
"""

query(first_fix_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,signup_page_visitors,first_fix_requests,first_fix_conv_rate,signup_page_visitors_per_day
0,2026-08-01 00:00:00.000,21,224298,23555,0.105017,10680
1,2026-07-01 00:00:00.000,31,295522,30197,0.102182,9532
2,2026-06-01 00:00:00.000,30,225103,22707,0.100874,7503
3,2026-05-01 00:00:00.000,31,269030,30226,0.112352,8678
4,2026-04-01 00:00:00.000,30,288653,33199,0.115014,9621
5,2026-03-01 00:00:00.000,31,368601,42812,0.116147,11890
6,2026-02-01 00:00:00.000,28,335632,36133,0.107657,11986
7,2026-01-01 00:00:00.000,31,358778,38036,0.106015,11573


First Fix conversion sits around **10–12%** of new signup-page visitors across the months observed. The most recent fully-observed month is used as the baseline for sizing, since it's the freshest complete read on current funnel behavior.

## Summary

Each step above added exactly one CTE, so this final query is a direct extension of Steps 1–3: the same eligibility gate (first web signup-page visit per visitor per month, excluding visitors who had already converted before that visit), with `month_days` added purely to turn the monthly totals into a daily rate. This is the exact query and baseline feeding the sample-size and duration calculations for this experiment.